# Modeling


In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# -------------------------
# Load
# -------------------------
df = pd.read_parquet("../data/processed/fraud_ecommerce.parquet")

# Drop raw datetime columns (we already engineered time features)
for c in ["signup_time", "purchase_time"]:
    if c in df.columns:
        df = df.drop(columns=[c])

# Drop columns that are not features / cause leakage / or huge-cardinality
DROP_ALWAYS = [
    "lower_bound_ip_address",
    "upper_bound_ip_address",
]
DROP_HIGH_CARD = [
    "device_id",
    "ip_address",
    "user_id",  # high-cardinality -> massive one-hot
]

for c in DROP_ALWAYS + DROP_HIGH_CARD:
    if c in df.columns:
        df = df.drop(columns=[c])

# Geo cleanup (if present)
if "country" in df.columns:
    df["geo_missing"] = df["country"].isna().astype(int)
    df["country"] = df["country"].fillna("Unknown")

# -------------------------
# Features/target
# -------------------------
target = "class"
X = df.drop(columns=[target])
y = df[target]

# -------------------------
# Split (stratified)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train class distribution:")
print(y_train.value_counts())

# -------------------------
# Columns
# -------------------------
# Keep only low-cardinality categoricals (everything else treated as numeric)
low_card_cats = [c for c in ["source", "browser", "sex", "country"] if c in X_train.columns]
num_cols = [c for c in X_train.columns if c not in low_card_cats]

print("\nCategorical cols used:", low_card_cats)
print("Numeric cols used:", num_cols)

# -------------------------
# Preprocess pipelines
# -------------------------
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, low_card_cats),
    ],
    remainder="drop"
)

# -------------------------
# Model pipeline: preprocess -> SMOTE -> Logistic Regression
# -------------------------
pipe = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=2000)),
])

pipe.fit(X_train, y_train)

# -------------------------
# Evaluate
# -------------------------
proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

auc_pr = average_precision_score(y_test, proba)
f1 = f1_score(y_test, pred)
cm = confusion_matrix(y_test, pred)

print(f"\nAUC-PR: {auc_pr:.4f}")
print(f"F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, pred))


Train class distribution:
class
0    109568
1     11321
Name: count, dtype: int64

Categorical cols used: ['source', 'browser', 'sex', 'country']
Numeric cols used: ['purchase_value', 'age', 'ip_int', 'time_since_signup_sec', 'hour_of_day', 'day_of_week', 'is_weekend', 'negative_time_since_signup', 'user_txn_count_1h', 'user_txn_count_24h', 'user_txn_index', 'geo_missing']

AUC-PR: 0.3954
F1: 0.2767
Confusion Matrix:
 [[17950  9443]
 [  859  1971]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.66      0.78     27393
           1       0.17      0.70      0.28      2830

    accuracy                           0.66     30223
   macro avg       0.56      0.68      0.53     30223
weighted avg       0.88      0.66      0.73     30223

